In [60]:
import json

# jsonl 파일 이름 (파일 경로에 맞게 수정)
phase = 2
candidates_num = 5
model = ['o3-mini', 'gem-f', 'gem-f-l']
model_name = model[1]

filename = f'evaluation/ours-result-p{phase}-{model_name}-{candidates_num}'
filename = f'evaluation/baseline-result-p{phase}-{model_name}-{candidates_num}'

filename = f'evaluation/ours-result-p{phase}-{model_name}-{candidates_num}.txt'
filename = f'evaluation/baseline-result-p{phase}-{model_name}-{candidates_num}.txt'
print(filename)

total_count = 0
tool_true_count = 0
arguments_true_count = 0

# answer.tool 별 성능을 저장할 딕셔너리
# 구조: {tool_name: {"total": 총개수, "tool_true": true 개수, "arguments_true": true 개수}}
performance = {}

len_idx = 0
with open(filename, "r", encoding="utf-8") as f:
    for line in f:
        len_idx += 1
        line = line.strip()
        if not line:
            continue
        data = json.loads(line)
        total_count += 1
        
        # 전체 results.tool, results.arguments 카운트
        if data.get("results", {}).get("tool") == True:
            tool_true_count += 1
        if data.get("results", {}).get("arguments") == True:
            arguments_true_count += 1

        # answer.tool 이름에 따른 성능 집계
        answer_tool = data.get("answer", {}).get("name")        
        if answer_tool is not None:            
            if answer_tool not in performance:
                performance[answer_tool] = {"total": 0, "tool_true": 0, "arguments_true": 0}
            performance[answer_tool]["total"] += 1
            if data.get("results", {}).get("tool") == True:
                performance[answer_tool]["tool_true"] += 1
            if data.get("results", {}).get("arguments") == True:
                performance[answer_tool]["arguments_true"] += 1

# 전체 비율 계산
overall_tool_ratio = tool_true_count / total_count if total_count > 0 else 0
overall_arguments_ratio = arguments_true_count / total_count if total_count > 0 else 0

print(f"전체 결과 대비 results.tool 성공률: {overall_tool_ratio:.4f}")
print(f"전체 결과 대비 results.arguments 성공률: {overall_arguments_ratio:.4f}\n")
print(len_idx)
# answer.tool 이름별 성능 출력
# print("answer.tool 이름별 성능:")
# for tool, counts in performance.items():    
#     total = counts["total"]
#     tool_ratio = counts["tool_true"] / total if total > 0 else 0
#     arguments_ratio = counts["arguments_true"] / total if total > 0 else 0
#     print(f"- {tool}:")
#     print(f"    results.tool 성공률: {tool_ratio:.2f}")
#     print(f"    results.arguments 성공률: {arguments_ratio:.2f}")


evaluation/baseline-result-p2-gem-f-5.txt
전체 결과 대비 results.tool 성공률: 0.9733
전체 결과 대비 results.arguments 성공률: 0.7800

150


In [120]:
import json
import pandas as pd

phase = 2
candidates_num = 10
model = ['o3-mini', 'gem-f', 'gem-f-l']
model_name = model[2]
filename_A = f'evaluation/baseline-result-p{phase}-{model_name}-{candidates_num}.txt'
filename_B = f'evaluation/ours-result-p{phase}-{model_name}-{candidates_num}.txt'

# 상태 판별 함수: results의 tool, arguments가 모두 True이면 success, 아니면 fail
def get_status(results):
    if results.get("tool") is True and results.get("arguments") is True:
        return "success"
    else:
        return "fail"

# 파일에서 레코드를 읽어서, 같은 unique_idx가 있으면 "_duplicated" 접미사를 붙여 별도의 row로 저장
def read_file(filename, is_baseline=True):
    data_dict = {}
    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            data = json.loads(line)
            uid = data["unique_idx"]
            status = get_status(data.get("results", {}))
            if is_baseline:
                record = {
                    "unique_idx": uid,
                    "conv_history": data.get("conversation_history"),
                    "query": data.get("query"),
                    "api_result_baseline": data["results"]["tool"],
                    "generated_baseline": data["generated"]["answer"]["tool"],
                    "parameters_result_baseline": data["results"]["arguments"],
                    "parameters_generated_baseline": data["generated"]["answer"]["parameters"],
                    "api_answer": data.get("answer", {}).get("name", "UNKNOWN"),
                    "parameters_answer": data.get("answer", {}).get("arguments"),
                    "baseline_status": status
                }
            else:
                record = {
                    "unique_idx": uid,
                    "rewrited_query": data.get("rewrited_query"),
                    "api_result_ours": data["results"]["tool"],
                    "generated_ours": data["generated"]["answer"]["tool"],
                    "parameters_result_ours": data["results"]["arguments"],
                    "parameters_generated_ours": data["generated"]["answer"]["parameters"],
                    "api_answer": data.get("answer", {}).get("name", "UNKNOWN"),
                    "parameters_answer": data.get("answer", {}).get("arguments"),
                    "ours_status": status
                }
            # 동일한 unique_idx가 이미 있으면, 새로운 키를 생성하여 추가
            if uid in data_dict:
                # 기본적으로 "uid_duplicated"를 사용하고, 이미 존재하면 번호를 붙임
                new_uid = uid + "_duplicated"
                counter = 1
                while new_uid in data_dict:
                    counter += 1
                    new_uid = uid + f"_duplicated_{counter}"
                # 새롭게 생성된 unique_idx를 record에 업데이트하고 저장
                record["unique_idx"] = new_uid
                data_dict[new_uid] = record
            else:
                data_dict[uid] = record
    return data_dict

# Baseline과 Ours 파일 모두 읽기 (성공/실패 모두 포함)
baseline_data = read_file(filename_A, is_baseline=True)
ours_data = read_file(filename_B, is_baseline=False)

# 두 파일의 데이터를 unique_idx 기준으로 병합
all_uids = set(baseline_data.keys()) | set(ours_data.keys())
merged_rows = []

for uid in all_uids:
    row = {"unique_idx": uid}
    b = baseline_data.get(uid, {})
    o = ours_data.get(uid, {})
    
    # Baseline 관련 값 (없으면 None)
    row["conv_history"] = b.get("conv_history")
    row["query"] = b.get("query")
    row["api_result_baseline"] = b.get("api_result_baseline")
    row["generated_baseline"] = b.get("generated_baseline")
    row["parameters_result_baseline"] = b.get("parameters_result_baseline")
    row["parameters_generated_baseline"] = b.get("parameters_generated_baseline")
    row["baseline_status"] = b.get("baseline_status")
    
    # Ours 관련 값 (없으면 None)
    row["rewrited_query"] = o.get("rewrited_query")
    row["api_result_ours"] = o.get("api_result_ours")
    row["generated_ours"] = o.get("generated_ours")
    row["parameters_result_ours"] = o.get("parameters_result_ours")
    row["parameters_generated_ours"] = o.get("parameters_generated_ours")
    row["ours_status"] = o.get("ours_status")
    
    # 공통 key: 한쪽에서라도 값이 있으면 사용
    row["api_answer"] = b.get("api_answer") or o.get("api_answer")
    row["parameters_answer"] = b.get("parameters_answer") or o.get("parameters_answer")
    
    # error_category 결정
    bs = row.get("baseline_status")
    os = row.get("ours_status")
    if bs == "fail" and os == "fail":
        row["error_category"] = "fail_common"
    elif bs == "fail" and os == "success":
        row["error_category"] = "fail_baseline_only"   # Baseline 실패, Ours 성공
    elif bs == "success" and os == "fail":
        row["error_category"] = "fail_ours_only"         # Ours 실패, Baseline 성공
    elif bs == "success" and os == "success":
        row["error_category"] = "success"
    else:
        if bs is None and os is not None:
            row["error_category"] = "ours_only"
        elif os is None and bs is not None:
            row["error_category"] = "baseline_only"
        else:
            row["error_category"] = "unknown"
    
    merged_rows.append(row)

# 최종 DataFrame 생성
df_all = pd.DataFrame(merged_rows)

# error_category 기준으로 정렬 후, TSV 파일로 저장 (탭 구분자 사용, 인덱스 없이)
df_all_sorted = df_all.sort_values(by="error_category")
output_filename = "merged_data.tsv"
df_all_sorted.to_csv(output_filename, sep='\t', index=False)

print("Merged DataFrame (sorted by error_category) saved as TSV:")
print("\nTotal rows:", len(df_all_sorted))

# (이전 코드 생략: merged DataFrame(df_all_sorted) 생성 및 TSV 파일 저장 코드 이후)
df_all_sorted = df_all
# 전체 케이스 수
total = len(df_all_sorted)

# Baseline Accuracy (API, Parameter)
# Baseline의 결과가 True인 경우를 정답으로 간주
baseline_api_correct = (df_all_sorted["api_result_baseline"] == True).sum()
baseline_param_correct = (df_all_sorted["parameters_result_baseline"] == True).sum()
baseline_api_accuracy = baseline_api_correct / total
baseline_param_accuracy = baseline_param_correct / total

# Ours Accuracy (API, Parameter)
ours_api_correct = (df_all_sorted["api_result_ours"] == True).sum()
ours_param_correct = (df_all_sorted["parameters_result_ours"] == True).sum()
ours_api_accuracy = ours_api_correct / total
ours_param_accuracy = ours_param_correct / total

# Common Accuracy (API, Parameter)
# Common에서는 Baseline와 Ours 모두 True여야 정답으로 간주
common_api_correct = df_all_sorted.apply(lambda row: row.get("api_result_baseline") is True and row.get("api_result_ours") is True, axis=1).sum()
common_param_correct = df_all_sorted.apply(lambda row: row.get("parameters_result_baseline") is True and row.get("parameters_result_ours") is True, axis=1).sum()
common_api_accuracy = common_api_correct / total
common_param_accuracy = common_param_correct / total

print("Performance Metrics:")
print("---------------------")
print("Baseline Accuracy:")
print("  API Accuracy:       {:.2%}".format(baseline_api_accuracy))
print("  Parameter Accuracy: {:.2%}".format(baseline_param_accuracy))
print()
print("Ours Accuracy:")
print("  API Accuracy:       {:.2%}".format(ours_api_accuracy))
print("  Parameter Accuracy: {:.2%}".format(ours_param_accuracy))
print()
print("Common Accuracy:")
print("  API Accuracy:       {:.2%}".format(common_api_accuracy))
print("  Parameter Accuracy: {:.2%}".format(common_param_accuracy))


Merged DataFrame (sorted by error_category) saved as TSV:

Total rows: 150
Performance Metrics:
---------------------
Baseline Accuracy:
  API Accuracy:       92.67%
  Parameter Accuracy: 72.67%

Ours Accuracy:
  API Accuracy:       94.00%
  Parameter Accuracy: 80.67%

Common Accuracy:
  API Accuracy:       90.67%
  Parameter Accuracy: 69.33%


In [102]:
df_all.to_csv("evaluation_log.tsv", sep='\t')

In [113]:
import json

filename_A = 'evaluation/baseline-result-p2-gem-f-l-10.txt'
filename_B = 'evaluation/ours-result-p2-gem-f-l-10.txt'

# 파일 A에서 unique_idx 추출
unique_ids_A = set()
with open(filename_A, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue        
        data = json.loads(line)
        unique_ids_A.add(data["unique_idx"])

# 파일 B에서 unique_idx 추출
unique_ids_B = set()
with open(filename_B, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        data = json.loads(line)
        unique_ids_B.add(data["unique_idx"])

# 각각의 unique_idx 개수 출력
print("파일 A unique_idx 개수:", len(unique_ids_A))
print("파일 B unique_idx 개수:", len(unique_ids_B))

# 공통 unique_idx 구하기
common_ids = unique_ids_A.intersection(unique_ids_B)
print("공통 unique_idx 개수:", len(common_ids))
#print("공통 unique_idx 값들:", common_ids)


파일 A unique_idx 개수: 141
파일 B unique_idx 개수: 141
공통 unique_idx 개수: 141


In [109]:
import json

def find_duplicates(filename):
    # unique_idx별로 데이터를 저장할 딕셔너리 (key: unique_idx, value: 해당 레코드 리스트)
    records_by_uid = {}
    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            data = json.loads(line)
            uid = data["unique_idx"]
            records_by_uid.setdefault(uid, []).append(data)
    # 중복된 unique_idx (리스트 길이가 2 이상인 경우)만 필터링
    duplicates = {uid: recs for uid, recs in records_by_uid.items() if len(recs) > 1}
    return duplicates

# 파일 경로 설정
filename_A = 'evaluation/baseline-result-p2-gem-f-5.txt'
filename_B = 'evaluation/ours-result-p2-gem-f-5.txt'

# 각 파일에서 중복된 unique_idx 찾아보기
dup_A = find_duplicates(filename_A)
dup_B = find_duplicates(filename_B)

# # 결과 출력
# print("파일 A에서 중복된 unique_idx와 데이터:")
# for uid, records in dup_A.items():
#     print(f"unique_idx: {uid}")
#     for rec in records:
#         print(json.dumps(rec, ensure_ascii=False, indent=2))
#     print("-" * 40)

# print("\n파일 B에서 중복된 unique_idx와 데이터:")
# for uid, records in dup_B.items():
#     print(f"unique_idx: {uid}")
#     for rec in records:
#         print(json.dumps(rec, ensure_ascii=False, indent=2))
#     print("-" * 40)
